In [1]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import llm_tools as llt
from utils import is_casenum


In [2]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 4000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 500
REQUESTS_PER_BATCH = 1000

In [3]:
MIN_IDX = 0
MAX_IDX = 0
REPLACE = False
REPLACE_OUTPUT_FILE = True
MODE = 'ESTIMATE'  # ESTIMATE | BATCH | WRITE

In [4]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [5]:
PROMPT = Template("""
You are a document analyst processing supplemental documents submitted to the Los Angeles City Planning Commission. Your task is to read a meeting agenda and a supplemental document, then extract structured metadata about the supplemental document.

Return valid JSON only. Do not include any other text, markdown formatting, or code fences.

## Output Format

Here is an example of the expected output:

{
    "summary": "A letter from the Silver Lake Neighborhood Council opposing the proposed mixed-use development at 1234 Sunset Blvd. The council argues the project's height exceeds what is permitted under the community plan and requests the Commission deny the appeal.",
    "referenced_agenda_items": ["5a"],
    "document_type": "LETTER OR PETITION",
    "author_type": "NEIGHBORHOOD COUNCIL",
    "support_or_oppose": {
        "5a": "DEFINITELY OPPOSE"
    }
}

## Field Definitions

**summary** (string): A 2-4 sentence summary of the supplemental document and its relation to any items on the agenda.

**referenced_agenda_items** (list of strings): The agenda items that the supplemental document references. Use the agenda item numbers exactly as they appear on the agenda (e.g., "5a", "5b", "6", "7"). If the document does not reference any specific agenda item, return an empty list [].

**document_type** (string): Classify the document using one of the following categories:
- LETTER OR PETITION — A letter, petition, or public comment that expresses a position on an agenda item.
- STAFF REPORT — A report prepared by city planning staff.
- TECHNICAL MODIFICATION — A modification or correction to a previously filed report or condition of approval.
- LEGAL FILING — A legal brief, motion, or formal legal submission.
- ENVIRONMENTAL REVIEW — An environmental impact report, assessment, or related document.
- SCIENTIFIC OR TECHNICAL STUDY — A study, analysis, or data report of a scientific or technical nature.
- PROJECT APPLICATION MATERIALS — Plans, renderings, project descriptions, or other materials submitted by or on behalf of the applicant.
- ADMINISTRATIVE — Procedural documents such as continuance requests, withdrawal notices, or errata.
- OTHER — Use only if none of the above categories fit. Include a brief note in the summary explaining the document type.

**author_type** (string): Classify the author using one of the following categories:
- INDIVIDUAL — A member of the public with no stated formal affiliation.
- PUBLIC OFFICIAL — An elected official or representative of a government agency.
- NEIGHBORHOOD COUNCIL — A neighborhood council or its representative.
- ADVOCACY ORGANIZATION — A nonprofit, community group, or advocacy organization.
- DEVELOPER — A property owner, developer, or their company.
- LAWYER — An attorney or law firm, whether representing a developer, community group, or other party.
- PLANNING STAFF — City planning department staff.
- CONSULTANT — A paid consultant or technical expert (e.g., traffic engineer, environmental consultant).
- MULTIPLE AUTHORS — A document with multiple authors of different types. Note the author types in the summary.
- OTHER — Use only if none of the above categories fit. Note the author type in the summary.

**support_or_oppose** (dict): For each item listed in referenced_agenda_items, indicate the document's position. The keys must exactly match the strings in referenced_agenda_items. Use one of the following values:
- DEFINITELY SUPPORT — The document clearly advocates for approval of the item.
- SOMEWHAT SUPPORT — The document is generally favorable but includes caveats, conditions, or hedging.
- NEUTRAL — The document does not express a clear position for or against.
- SOMEWHAT OPPOSE — The document is generally unfavorable but includes some acknowledgment of merit or willingness to negotiate.
- DEFINITELY OPPOSE — The document clearly advocates for denial of the item.
- N/A — The document is purely technical or informational and does not express a position.

If the document does not reference any specific agenda items, return an empty dict {}.

## Classification Guidelines

1. Start by reviewing all agenda items to understand the full set of cases, addresses, and case numbers under consideration. When determining support or oppose, give precedence to the agenda's framing of each item rather than relying solely on the document's self-description.

2. Match the supplemental document to agenda items based on any of: case number, agenda item number, project name, or street address.

3. Most supplemental documents reference only one agenda item, but some may reference multiple items or none at all.

4. When classifying support_or_oppose, consider the practical effect of what the document advocates. A document may express general support for development but oppose the specific project under review, or vice versa. Classify based on whether the document's requested action would likely lead to approval or denial of the agenda item.

5. If the document is corrupted, illegible, blank, a cover page, a section heading, or otherwise unusable, set summary to a brief explanation of why it is unusable, referenced_agenda_items to [], document_type to "N/A", author_type to "N/A", and support_or_oppose to {}.

<agenda>

$agenda_text

</agenda>

<supplemental_document>

$document_text

</supplemental_document>
""")

In [ ]:
# write prompt to figures
with open(os.path.join(LOCAL_PATH, 'figures', 'summarize_supplemental_docs_prompt.tex'), 'w') as f:
    out = PROMPT
    out = out.strip()
    out = out.replace('\n', '\\\\ \n')
    f.write(out)

KeyError: 'agenda_text'

In [ ]:
input_tokens = 0
output_tokens = 0
n_requests = 0
n_direct = 0
batch_num = 0
max_input_token_length = 0
max_input_token_date = ""
max_input_token_prompt = ""

rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)

t0 = time.time()
prompts_to_submit = []
for idx, row in meetings_df.iterrows():
    if idx < MIN_IDX or idx > MAX_IDX:
        continue

    date = row['date']
    year = date[0:4]

    print(f"{idx} {date} ...")

    # Agenda data
    agenda_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized.json")
    agenda_override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-override.json")
    if (not os.path.exists(agenda_filepath)) and (not os.path.exists(agenda_override_filepath)):
        print("No agenda text file found, skipping.")
        continue
    if os.path.exists(agenda_override_filepath):
        with open(agenda_override_filepath, 'r', encoding='utf-8') as f:
            j = json.load(f)
    else:
        with open(agenda_filepath, 'r', encoding='utf-8') as f:
            j = json.load(f)
    
    # Supplemental documents
    docs_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs.pkl")
    if not os.path.exists(docs_filepath):
        print("No supplemental documents file found, skipping.")
        continue
    docs_df = pd.read_pickle(docs_filepath)

    # Output paths
    output_dir = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date)
    os.makedirs(output_dir, exist_ok=True)
    output_filepath = os.path.join(output_dir, "supplemental-docs-summarized.parquet")
    if (not REPLACE_OUTPUT_FILE) and os.path.exists(output_filepath):
        print("Output file exists, skipping.")
        continue


batch_db_conn.close()
response_db_conn.close()


0 2018-05-10 ...
    wrote output file
1 2018-05-23 ...
    wrote output file
2 2018-06-14 ...
    wrote output file
3 2018-07-12 ...
    wrote output file
4 2018-07-26 ...
    wrote output file
5 2018-08-09 ...
    wrote output file
6 2018-08-23 ...
    wrote output file
7 2018-09-13 ...
    wrote output file
8 2018-09-27 ...
    wrote output file
9 2018-10-11 ...
    wrote output file
10 2018-10-25 ...
    wrote output file
11 2018-11-08 ...
    wrote output file
12 2018-11-29 ...
    wrote output file
13 2018-12-13 ...
    wrote output file
14 2018-12-20 ...
    wrote output file
15 2019-01-10 ...
    wrote output file
16 2019-01-24 ...
    wrote output file
17 2019-02-14 ...
    wrote output file
18 2019-02-28 ...
    wrote output file
19 2019-03-14 ...
    wrote output file
20 2019-03-28 ...
    wrote output file
21 2019-04-11 ...
    wrote output file
22 2019-05-09 ...
    wrote output file
23 2019-05-23 ...
    wrote output file
24 2019-06-13 ...
    wrote output file
25 2019-06

In [ ]:
# Cost Estimate

total_input_cost = input_tokens / 1e6 * input_cost
total_output_cost = output_tokens / 1e6 * output_cost

print(f"Estimated number of requests: {n_requests:,.0f}")
print(f"Estimated total input tokens: {input_tokens:,.0f}")
print(f"Estimated total input cost: ${total_input_cost:.2f}")
print(f"Estimated total output tokens: {n_requests * output_tokens_per_request:,.0f}")
print(f"Estimated total output cost: ${total_output_cost:.2f}")
print(f"Estimated total cost: ${total_input_cost + total_output_cost:.2f}")
print(f"Max input token length: {max_input_token_length:,.0f} on date {max_input_token_date}")

Estimated number of requests: 0
Estimated total input tokens: 0
Estimated total input cost: $0.00
Estimated total output tokens: 0
Estimated total output cost: $0.00
Estimated total cost: $0.00
Max input token length: 0 on date 
